# 01 — Exploratory Data Analysis

**Project:** Predicting 30-Day Hospital Readmission  
**Dataset:** `hospital_readmission_risk_dataset_2026_v1_18000rows.csv`  
**Phase:** 1 — Data Understanding

This notebook loads the raw dataset, inspects its structure, and surfaces patterns that will guide cleaning and feature-engineering decisions.

| # | Section |
|---|---|
| 1 | Environment setup |
| 2 | Load config + raw data |
| 3 | Initial inspection |
| 4 | Missing-value analysis |
| 5 | Class (outcome) distribution |
| 6 | Numeric feature distributions |
| 7 | Categorical feature distributions |
| 8 | Relationships with the target |
| 9 | Correlation matrix |
| 10 | Takeaways |

---
## 1 — Environment setup

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Make src/ importable when running from notebooks/
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, get_logger, set_seed

logger = get_logger('01_eda')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
FIGURES = ROOT / 'reports' / 'figures' / 'eda'
FIGURES.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print('Environment ready.')

---
## 2 — Load config and raw data

In [ ]:
config  = load_config(ROOT / 'config' / 'config.yaml')
set_seed(config['random_seed'])

TARGET   = config['data']['target_column']
raw_path = ROOT / config['paths']['raw_data']

print(f'Raw data : {raw_path}')
print(f'Target   : {TARGET}')

In [ ]:
from src.data_preparation import load_raw_data

df = load_raw_data(raw_path)
df.head()

---
## 3 — Initial inspection

In [ ]:
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n')
df.info()

In [ ]:
df.describe(include='all').T

---
## 4 — Missing-value analysis

This dataset has **no missing values**, but we still surface any would-be issues here so the notebook is reusable with other datasets.

In [ ]:
missing = (
    df.isna().sum()
      .rename('n_missing')
      .to_frame()
      .assign(pct_missing=lambda x: x['n_missing'] / len(df) * 100)
      .query('n_missing > 0')
      .sort_values('pct_missing', ascending=False)
)

if missing.empty:
    print('No missing values found. ✓')
else:
    display(missing)
    fig, ax = plt.subplots(figsize=(10, max(3, len(missing) * 0.4)))
    missing['pct_missing'].plot.barh(ax=ax, color='steelblue')
    ax.set_xlabel('% Missing')
    ax.set_title('Missing Values by Column')
    plt.tight_layout()
    plt.savefig(FIGURES / 'missing_values.png', bbox_inches='tight')
    plt.show()

### Data-quality note — `Creatinine_Level`

Although there are no `NaN` values, `Creatinine_Level` contains **31 physically-impossible negative values** (min = –0.42 mg/dL).  
These are clipped to 0.0 in `clean_data()` via the `clip_values` config key.

In [ ]:
neg_creatinine = (df['Creatinine_Level'] < 0).sum()
print(f"Creatinine_Level < 0: {neg_creatinine} rows  (will be clipped to 0.0 in clean_data)")

---
## 5 — Class (outcome) distribution

> **Key finding:** This synthetic dataset has a **74% positive rate** — the *majority* class is readmitted.  
> This is the inverse of real-world data (~15–25%).  
> For modelling: treat class 0 (not readmitted) as the minority; use `class_weight='balanced'` or set `scale_pos_weight ≈ 0.35`.

In [ ]:
counts = df[TARGET].value_counts().sort_index()
rates  = counts / counts.sum() * 100

summary = pd.DataFrame({'Count': counts, 'Pct (%)': rates.round(1)})
summary.index.name = TARGET
print('Class distribution:')
display(summary)
print(f'\nImbalance ratio (majority/minority): {counts.max() / counts.min():.1f}x')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count bar
bar_colors = sns.color_palette('muted', 2)
axes[0].bar(counts.index.astype(str), counts.values, color=bar_colors)
axes[0].set_title('Readmission Counts')
axes[0].set_xlabel(f'{TARGET} (0 = No, 1 = Yes)')
axes[0].set_ylabel('Count')
for i, v in zip(counts.index, counts.values):
    axes[0].text(str(i), v + counts.max() * 0.01, f'{v:,}', ha='center')

# Pie
labels = [f'Not Readmitted\n({rates[0]:.1f}%)', f'Readmitted\n({rates[1]:.1f}%)']
axes[1].pie(counts.values, labels=labels, colors=bar_colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Readmission Proportion')

plt.suptitle('Target Variable Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'class_distribution.png', bbox_inches='tight')
plt.show()

---
## 6 — Numeric feature distributions

All numeric columns, excluding the target.  
Look for: heavy skew, bimodal distributions, suspicious spikes at round numbers.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.difference([TARGET]).tolist()
print(f'Numeric columns ({len(numeric_cols)}): {numeric_cols}')

n_cols = 4
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col].dropna(), ax=axes[i], kde=True, bins=30, color='steelblue')
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'numeric_distributions.png', bbox_inches='tight')
plt.show()

---
## 7 — Categorical feature distributions

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.difference([TARGET]).tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

n_cols = 2
n_rows = int(np.ceil(len(cat_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3.5))
axes = np.array(axes).flatten()

for i, col in enumerate(cat_cols):
    top_vals = df[col].value_counts().head(15)
    axes[i].barh(top_vals.index.astype(str)[::-1], top_vals.values[::-1],
                 color=sns.color_palette('muted')[i % 6])
    axes[i].set_title(col)
    axes[i].set_xlabel('Count')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'categorical_distributions.png', bbox_inches='tight')
plt.show()

---
## 8 — Relationships with the target

Box plots show how each numeric feature's distribution differs between readmitted (1) and not-readmitted (0) patients.

In [ ]:
# Select the most clinically meaningful numeric features to plot
priority_cols = [
    'Comorbidity_Index', 'Chronic_Disease_Count', 'Severity_Score',
    'Length_of_Stay', 'Previous_Admissions_6M', 'Medication_Adherence_Score',
    'Age', 'HbA1c_Level', 'Number_of_Medications'
]
plot_cols = [c for c in priority_cols if c in df.columns] or numeric_cols[:9]

n_cols = 3
n_rows = int(np.ceil(len(plot_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
axes = np.array(axes).flatten()

for i, col in enumerate(plot_cols):
    sns.boxplot(data=df, x=TARGET, y=col, ax=axes[i],
                palette='muted', showfliers=False)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('Readmitted (0=No, 1=Yes)')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Key Features vs. Readmission Outcome', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'features_vs_target.png', bbox_inches='tight')
plt.show()

### Readmission rate by categorical feature

In [ ]:
if cat_cols:
    n_cols = 2
    n_rows = int(np.ceil(len(cat_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3.5))
    axes = np.array(axes).flatten()

    for i, col in enumerate(cat_cols):
        rates_by_cat = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
        axes[i].barh(rates_by_cat.index.astype(str)[::-1], rates_by_cat.values[::-1],
                     color=sns.color_palette('muted')[i % 6])
        axes[i].set_title(f'Readmission rate by {col}')
        axes[i].set_xlabel('Readmission Rate')
        axes[i].axvline(df[TARGET].mean(), color='red', linestyle='--',
                        linewidth=0.8, label='Overall avg')
        axes[i].legend(fontsize=7)
        axes[i].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Readmission Rate by Categorical Feature', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES / 'readmission_rate_by_category.png', bbox_inches='tight')
    plt.show()

---
## 9 — Correlation matrix

Flag feature pairs with |r| > 0.85 as potential multicollinearity issues.

In [ ]:
corr_cols = numeric_cols + [TARGET]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(max(10, len(corr_cols)), max(8, len(corr_cols) - 1)))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.4,
            annot_kws={'size': 7})
ax.set_title('Numeric Feature Correlation Matrix (incl. target)')
plt.tight_layout()
plt.savefig(FIGURES / 'correlation_matrix.png', bbox_inches='tight')
plt.show()

# Flag high correlations (excluding target row)
feature_corr = df[numeric_cols].corr()
high_corr = (
    feature_corr.abs()
       .where(np.tril(np.ones(feature_corr.shape), k=-1).astype(bool))
       .stack()
       .reset_index()
       .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'abs_corr'})
       .query('abs_corr > 0.85')
       .sort_values('abs_corr', ascending=False)
)
if high_corr.empty:
    print('No feature pairs with |r| > 0.85 — no multicollinearity concern.')
else:
    print('High-correlation pairs (|r| > 0.85):')
    display(high_corr)

### Target correlations (strongest predictors)

In [ ]:
target_corr = (
    df[numeric_cols + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(8, max(4, len(target_corr) * 0.35)))
colors = ['tomato' if v < 0 else 'steelblue' for v in target_corr.values]
ax.barh(target_corr.index[::-1], target_corr.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.7)
ax.set_xlabel(f'Pearson r with {TARGET}')
ax.set_title('Feature Correlations with Target (strongest first)')
plt.tight_layout()
plt.savefig(FIGURES / 'target_correlations.png', bbox_inches='tight')
plt.show()

print('Top 5 positive:')
print(target_corr.head(5).to_string())
print('\nTop 5 negative:')
print(target_corr.tail(5).to_string())

---
## 10 — Takeaways

### Dataset summary
| Property | Value |
|---|---|
| Rows | 18,000 |
| Columns | 25 |
| Missing values | None |
| Target | `Readmitted_Within_30_Days` |
| Positive rate | ~74.2% (readmitted = majority class) |
| Imbalance ratio | ~2.9x |

### Data-quality issues
- **Creatinine_Level**: 31 rows with negative values → clipped to 0 in `clean_data`
- No other impossible values detected

### Class imbalance
- The majority class (1 = readmitted) is ~74%.  Use `class_weight='balanced'` or `scale_pos_weight ≈ 0.35` in XGBoost.
- Prioritise **Precision-Recall AUC** over ROC-AUC given the imbalance.

### Feature notes
- No individual binary comorbidity flags; use `Comorbidity_Index` and `Chronic_Disease_Count` directly.
- `Discharge_Disposition` has 3 values: Home, Nursing Facility, Rehab — mapped to Home/Facility in `discharge_disposition_cat`.
- Binary flags: `ICU_Stay_Flag`, `High_Risk_Medication_Flag`, `Followup_Appointment_Scheduled`.

### Next steps
1. Run `02_cleaning_check.ipynb` to verify clean + feature-engineered output.
2. Proceed to `03_modeling_baseline.ipynb`.